# Drive - Basic operations

This notebook will cover performing basic H2O Drive operations via the Drive Python client.

The topics we'll cover include:
- Selecting a bucket
- Uploading
- Listing
- Downloading
- Deleting

## Requirements and helpers

Let's install the H2O Drive Python Client.

In [ ]:
import sys
!{sys.executable} -m pip install -q "h2o_drive>=4.0.0"

Let's create a local `books.csv` file which we'll later use to demonstrate uploads.

In [ ]:
with open("books.csv", "w") as f:
    f.write("Title, Author, Year\n")
    f.write("Pride and Prejudice, Jane Austen, 1813\n")
    f.write("Frankenstein, Mary Shelley, 1818\n")
    f.write("Of Mice and Men, John Steinbeck, 1937\n")
    f.write("The Catcher in the Rye, J.D. Salinger, 1945\n")

We'll define a helper to use later on.

In [ ]:
from typing import List
import h2o_drive

def print_objects(objs: List[h2o_drive.ObjectSummary]) -> None:
    """Neatly display a list of objects."""
    for o in objs:
        print(o.key)

## Terminology

H2O Drive is an object store. Object stores typically involve the concepts of **buckets**, **objects** and **keys**.

To explain these concepts, let's think of an object store like a filing cabinet, where each document in the cabinet is individually labelled.
- A **bucket** is the filing cabinet itself.
  - Users may be authorized to access certain filing cabinets, while not authorized to access others.
  - Some users may only be authorized to read documents in a filing cabinet while others may have authorization to add or remove documents.

- **Objects** are the individual documents in the filing cabinet.
- A **key** is a document's unique label (in this analogy, each document in the cabinet is individually labelled).
  - These identifiers uniquely identify a document in the file cabinet - no two documents can have the same label.

> 💡 Tip
>
> Be on the lookout for notes labelled "🗄️ Filing cabinet analogy".

> ℹ️ Note
>
> Please see the Drive notebook titled `Drive - Keys and prefixes` for more information about, and uses for, prefixes.

## Connecting to Drive

Let's connect to H2O Drive.

> ℹ️ Note
>
> For demonstration purposes, this tutorial connects to H2O Drive in the most convenient way. This works when run from within the H2O AI Cloud or locally when the H2O CLI is configured.
>
> If this is not the case, and the following code fails, please see the Drive notebook titled `Drive - Connecting from different environments` for a walkthrough on connecting to Drive from your environment.

In [ ]:
import h2o_drive

drive = h2o_drive.connect()

## Basic operations

### Selecting a bucket

`workspace_bucket()` returns the bucket associated with the specified workspace. It takes one argument:
- `workspace`: The name, or identifier, of the workspace for which to retrieve the associated bucket.

Now that we're connected to H2O Drive, we'll want to select a bucket to work with.

Every H2O workspace has a corresponding Drive bucket. Any user or service from across the platform, with access to that workspace, can store data in the workspace's associated Drive bucket. Buckets serve as storage for persisting data and enable platform-wide sharing and collaboration.

For this tutorial, let's work with the bucket associated with our personal H2O workspace. Users automatically recieve appropriate permissions to access their personal workspaces, which can be accessed via the special alias `default`.

> 🗄️ Filing cabinet analogy
>
> Saying that every workspace has its own bucket is analogous to saying that every workspace has its own filing cabinet. These filing cabinets are isolated from one another.

In [ ]:
bucket = drive.workspace_bucket("default")

### Uploading

`upload_file()` uploads a local file, resulting in a new object at the specified key in the bucket. It takes two arguments:
- `filename`: The file to upload. The contents of this file will become an object in the Drive bucket.
- `key`: The key at which to store the resulting object. In other words, the name attached to the object.

Let's upload our `books.csv` file twice. Once at a simple key (`example-books.csv`), and then again at a key which resembles a hierarchical structure (`example-directory/classic-books.csv`).

Although keys are all flat in nature, a hierarchical-_looking_ key can be used to suggest that the object is part of some logical `example-directory` collection.

> ☝️ Important
>
> Object operations are asynchronous. Don't forget to `await` these commands.

In [ ]:
await bucket.upload_file("books.csv", "example-books.csv")
await bucket.upload_file("books.csv", "example-directory/classic-books.csv")

The content of our local `books.csv` file is now stored as two different objects inside our bucket.

### Listing

`list_objects()` returns the list of objects in the bucket. It takes one optional argument:
- `prefix`: When specified, only objects at keys starting with the specified value are listed.

Let's list the objects currently in our bucket. We'll use the `print_objects()` helper we defined earlier to print object keys cleanly.

In [ ]:
objects = await bucket.list_objects()
print_objects(objects)

> ℹ️ Note
>
> You may notice more objects than just the ones we've created thus far.
>
> This is fine. H2O services may already be leveraging your Drive Bucket to store data.
>
> For the rest of this tutorial, just focus on the files relevant to our purposes.

Suppose we only want to list objects whose keys start with the prefix `example-directory/` to mimic listing only objects in some collection or directory.

We can specify this prefix as a filter when listing objects.

In [ ]:
objects = await bucket.list_objects(prefix="example-directory/")
print_objects(objects)

Note that the object is returned with its full key, including the `example-directory/` prefix which we filtered on.

Just because we filtered results to a specific prefix doesn't change the fact that the object's proper key includes that prefix.

### Downloading

`download_file()` downloads the object at the specified key and writes it to the specified local file. It takes two arguments:
- `key`: The key of the object to download.
- `filename`: The file, on the local filesystem, the object is written to.

Let's download one of the two objects we created in the previous section. We'll use `download_file()` to save the contents of the object as a file at the local path `./downloaded-books.csv`.

In [ ]:
await bucket.download_file("example-books.csv", "./downloaded-books.csv")

### Deleting

`delete_object()` deletes the object at the specified key. It takes a single argument:
- `key`: The key of the object to delete.

Let's delete one of the two objects we've created.

In [ ]:
await bucket.delete_object("example-books.csv")

Listing the objects of our Drive bucket will confirm that object is indeed gone.

In [ ]:
objects = await bucket.list_objects()
print_objects(objects)

## Cleanup

Let's remove our example objects and local files, leaving your Drive bucket and notebook environment in the state it was before we got started.

In [ ]:
await bucket.delete_object("example-books.csv")
await bucket.delete_object("example-directory/classic-books.csv")

import os
os.remove("./books.csv")
os.remove("./downloaded-books.csv")